# SHIELD — Random Forest: 7 financial-statement candidates versus 20 inputs

**Question:** does a Random Forest using only seven predefined financial-statement-based feature candidates retain useful bankruptcy detection compared with the historical20-input baseline?

This **one notebook** runs both feature sets using a stratified **80:20 split**, **10-fold ordinary training CV**, **seed42**, fixed RF parameters and **threshold0.5**. Start with **SMOTE0.25**, then change one setting to **1.0** and run again. Ratio-specific settings produce separate output folders.

No imputation, feature reconstruction, new IG selection, GridSearchCV or threshold tuning. The seven candidates were selected for potential accounting availability, not because they performed best on this test set. Historical20 selection and repeated holdout inspection mean all results remain **exploratory**, not independent final validation.

**Important:** the seven are candidate ratios derivable from company accounts. Their exact historical donor definitions and transformations are not yet established. This experiment uses existing dataset columns; it does not validate entering raw SME financial amounts into the prototype or prove validity for Malaysian SMEs. It does not modify the prototype.


## 1. Install modelling dependencies
Pinned modelling versions match the verified local comparison environment. If Colab already imported a different version, restart the runtime after installing, then continue from the settings cell.

In [ ]:
%pip install -q scikit-learn==1.9.0 imbalanced-learn==0.14.2 pandas numpy matplotlib openpyxl


## 2. Settings and data paths
Only change SMOTE_RATIO between0.25 and1.0. In Colab, update the Drive path if necessary, then run the remaining cells top to bottom. Use the original unresampled Excel file, not the SMOTE-expanded training files.

In [ ]:
from pathlib import Path
SMOTE_RATIO=0.25           # Second run: change to 1.0
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE=Path('/content/drive/MyDrive/SHIELD_Project')
except ImportError:
    BASE=Path.cwd()
    if BASE.name=='notebooks': BASE=BASE.parent
RAW_PATH=BASE/'data'/'bankruptcy_italian_companies_2023.xlsx'
OUTPUT_BASE=BASE/'reports'/'rf_seven_vs_twenty'
print('Data:',RAW_PATH)
print('SMOTE ratio:',SMOTE_RATIO)


## 3. Fixed features, RF and metrics

RF: **200 trees, max_depth10, min_samples_split2, min_samples_leaf1, max_features sqrt, no class weights**. SMOTE: k_neighbors5. Balancing occurs **inside each training fold**, never on validation or test companies. No new scaling is introduced, matching this controlled historical RF setup. Because max_features is sqrt, the absolute number of candidate split features changes with feature count.

The baseline is retrained in the same environment; compare against this fresh baseline rather than assuming historical scores will reproduce exactly. The1.0 arm uses identical RF parameters for isolation, not an automatic reproduction of the original best-tuned1.0 model.

The seven and their historical IG positions are shown after data validation. Historical scores are recorded, not recalculated. Some of these seven encode related leverage/profit information; they are not seven independent signals.


In [ ]:
import hashlib, json, importlib.metadata
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    fbeta_score, roc_auc_score, average_precision_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

FEATURES=['ROA(C) before interest and depreciation before interest', 'ROA(A) before interest and % after tax', 'ROA(B) before interest and depreciation after tax', 'Continuous interest rate (after tax)', 'Net Value Per Share (B)', 'Net Value Per Share (A)', 'Persistent EPS in the Last Four Seasons', 'Per Share Net profit before tax', 'Interest Expense Ratio', 'Debt ratio %', 'Net worth/Assets', 'Borrowing dependency', 'Net profit before tax/Paid-in capital', 'Retained Earnings to Total Assets', 'Net Income to Total Assets', "Net Income to Stockholder's Equity", 'Liability to Equity', 'Degree of Financial Leverage (DFL)', 'Interest Coverage Ratio (Interest expense to EBIT)', 'Equity to Liability']
SEVEN_FEATURES=["Net Income to Stockholder's Equity", 'Net Income to Total Assets', 'Retained Earnings to Total Assets', 'Debt ratio %', 'Net worth/Assets', 'Equity to Liability', 'Liability to Equity']
HISTORICAL_IG_RANKS={"Net Income to Stockholder's Equity": 5, 'Net Income to Total Assets': 7, 'Retained Earnings to Total Assets': 13, 'Debt ratio %': 14, 'Net worth/Assets': 16, 'Equity to Liability': 17, 'Liability to Equity': 20}
HISTORICAL_IG_SCORES={"Net Income to Stockholder's Equity": 0.036753, 'Net Income to Total Assets': 0.035144, 'Retained Earnings to Total Assets': 0.032738, 'Debt ratio %': 0.032479, 'Net worth/Assets': 0.032132, 'Equity to Liability': 0.031732, 'Liability to Equity': 0.03066}

def scenarios():
    return {'baseline_20':FEATURES.copy(),'practical_7':SEVEN_FEATURES.copy()}

def make_model(ratio,small=False):
    return Pipeline([('smote',SMOTE(sampling_strategy=ratio,k_neighbors=5,random_state=42)),
        ('model',RandomForestClassifier(n_estimators=5 if small else 200,max_depth=10,
        min_samples_split=2,min_samples_leaf=1,max_features='sqrt',class_weight=None,
        random_state=42,n_jobs=2))])

def score_metrics(y,score):
    pred=(np.asarray(score)>=.5).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {'Accuracy (%)':100*accuracy_score(y,pred),
        'Precision (%)':100*precision_score(y,pred,zero_division=0),
        'Recall (%)':100*recall_score(y,pred,zero_division=0),
        'F1-Score (%)':100*f1_score(y,pred,zero_division=0),
        'ROC-AUC (%)':100*roc_auc_score(y,score),
        'F2 (%)':100*fbeta_score(y,pred,beta=2,zero_division=0),
        'PR-AUC (AP) (%)':100*average_precision_score(y,score),
        'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp)}

def checked_predictions(path,rows,y):
    with np.load(path,allow_pickle=False) as saved:
        np.testing.assert_array_equal(saved['row_index'],rows)
        np.testing.assert_array_equal(saved['y_true'],np.asarray(y))
        score=saved['y_score'].copy()
    if len(score)!=len(y) or not np.isfinite(score).all():
        raise ValueError('Invalid prediction checkpoint: '+str(path))
    return score


## 4. Validate original data and split once
Company and Bankrupt are excluded from predictors. Both feature sets use the identical companies and folds. There must be no unresolved missing values or duplicate full20-input rows. No rows are silently deleted. The output folder records input hash, versions, parameters and scenario membership. Full20 validation is necessary for the paired baseline comparison.

In [ ]:
raw=pd.read_excel(RAW_PATH) if RAW_PATH.suffix.lower() in ['.xlsx','.xls'] else pd.read_csv(RAW_PATH,float_precision='round_trip')
raw.columns=raw.columns.str.strip()
if raw.columns.duplicated().any(): raise ValueError('Duplicate column names')
if 'Bankrupt' not in raw: raise ValueError('Bankrupt target required')
if 'Company' in raw and raw.Company.duplicated().any(): raise ValueError('Duplicate company IDs: investigate/group before splitting')
if any(f not in raw for f in FEATURES): raise ValueError('Input does not contain all 20 financial features')
X=raw[FEATURES].apply(pd.to_numeric,errors='raise')
y=pd.to_numeric(raw.Bankrupt,errors='raise')
if X.isna().any().any() or y.isna().any() or not np.isfinite(X.to_numpy()).all():
    raise ValueError('Supply complete original data. Scenarios remove columns; no imputation is performed.')
if set(y.unique())!={0,1}: raise ValueError('Target must contain 0=Not Bankrupt and 1=Bankrupt')
y=y.astype(int)
if X.duplicated().any(): raise ValueError('Duplicate full predictor rows: investigate before splitting')
train_idx,test_idx=train_test_split(np.arange(len(y)),test_size=.20,stratify=y,random_state=42)
X_train,y_train=X.iloc[train_idx],y.iloc[train_idx]
X_test,y_test=X.iloc[test_idx],y.iloc[test_idx]
if SMOTE_RATIO not in [.25,1.0]: raise ValueError('Choose SMOTE_RATIO=0.25 or 1.0')
SCENARIOS=scenarios()
versions={p:importlib.metadata.version(p) for p in ['numpy','pandas','scikit-learn','imbalanced-learn']}
config={'version':'shield-rf-seven-vs-twenty-v1','raw_sha256':hashlib.sha256(RAW_PATH.read_bytes()).hexdigest(),
    'feature_names':FEATURES,'scenarios':SCENARIOS,'smote_ratio':SMOTE_RATIO,
    'seed':42,'test_size':.20,'folds':10,'threshold':.5,'versions':versions,
    'rf':make_model(SMOTE_RATIO).named_steps['model'].get_params(),
    'smote_k_neighbors':5,'scaling':'none','feature_selection':'fixed historical20 versus predefined7; no IG reselection'}
serialized=json.dumps(config,sort_keys=True,default=str)
RUN_ID=hashlib.sha256(serialized.encode()).hexdigest()[:16]
OUT=OUTPUT_BASE/RUN_ID
OUT.mkdir(parents=True,exist_ok=True)
if (OUT/'run_config.json').exists() and (OUT/'run_config.json').read_text(encoding='utf-8')!=serialized:
    raise ValueError('Checkpoint settings mismatch')
(OUT/'run_config.json').write_text(serialized,encoding='utf-8')
folds=list(StratifiedKFold(n_splits=10,shuffle=True,random_state=42).split(X_train,y_train))
membership=pd.DataFrame({'row_index':np.arange(len(y)),'Bankrupt':y,
    'partition':np.where(np.isin(np.arange(len(y)),train_idx),'training','test'),'validation_fold':-1})
for fold,(_,va) in enumerate(folds,1): membership.loc[train_idx[va],'validation_fold']=fold
membership.to_csv(OUT/'partition_membership.csv',index=False)
manifest=[{'Scenario':name,'Features':len(cols),'Removed':'; '.join(c for c in FEATURES if c not in cols)} for name,cols in SCENARIOS.items()]
pd.DataFrame(manifest).to_csv(OUT/'scenario_manifest.csv',index=False)
print('Original training counts:',y_train.value_counts().sort_index().to_dict())
print('Original test counts:',y_test.value_counts().sort_index().to_dict())
print('Scenarios:',len(SCENARIOS),'| SMOTE ratio:',SMOTE_RATIO,'| Output:',OUT)
display(pd.DataFrame(manifest))

rank_table=pd.DataFrame({'Feature':SEVEN_FEATURES,
    'Historical IG rank':[HISTORICAL_IG_RANKS[f] for f in SEVEN_FEATURES],
    'Historical IG score':[HISTORICAL_IG_SCORES[f] for f in SEVEN_FEATURES]})
rank_table.to_csv(OUT/'seven_feature_historical_ig.csv',index=False)
display(rank_table)


## 5. Training cross-validation
22 fits per ratio:10 folds plus1 full-training refit for each of2 feature sets. Every training company gets exactly one OOF prediction. Mean/SD of fold F1 and pooled OOF metrics are distinct. Completed fold predictions are reused only for matching run settings. If these scores are used to select the final feature set/ratio, they become selection scores.

In [ ]:
rows=[]
for name,columns in SCENARIOS.items():
    directory=OUT/name
    directory.mkdir(exist_ok=True)
    oof=np.full(len(y_train),np.nan)
    fold_metrics=[]
    for fold,(tr,va) in enumerate(folds,1):
        checkpoint=directory/f'fold_{fold:02d}.npz'
        if checkpoint.exists():
            scores=checked_predictions(checkpoint,train_idx[va],y_train.iloc[va])
        else:
            pipe=make_model(SMOTE_RATIO)
            pipe.fit(X_train.iloc[tr][columns],y_train.iloc[tr])
            scores=pipe.predict_proba(X_train.iloc[va][columns])[:,1]
            np.savez_compressed(checkpoint,row_index=train_idx[va],y_true=y_train.iloc[va],y_score=scores)
        oof[va]=scores
        fold_metrics.append({'Fold':fold,**score_metrics(y_train.iloc[va],scores)})
        print(name,'fold',fold,'F1:',round(fold_metrics[-1]['F1-Score (%)'],2))
    assert np.isfinite(oof).all()
    pd.DataFrame(fold_metrics).to_csv(directory/'fold_metrics.csv',index=False)
    np.savez_compressed(directory/'training_oof.npz',row_index=train_idx,y_true=y_train,y_score=oof)
    details=next(m for m in manifest if m['Scenario']==name)
    row={**details,'Model':'RandomForest','Experiment':'80-20 + CV=10',
        'Stage':'Training OOF','Split':.20,'CV':10,'SMOTE':SMOTE_RATIO,
        **score_metrics(y_train,oof),
        'CV_F1_Mean (%)':float(np.mean([m['F1-Score (%)'] for m in fold_metrics])),
        'CV_F1_SD (%)':float(np.std([m['F1-Score (%)'] for m in fold_metrics],ddof=1))}
    rows.append(row)
    pd.DataFrame(rows).to_csv(OUT/'results.csv',index=False)
    report=classification_report(y_train,(oof>=.5).astype(int),labels=[0,1],target_names=['Not Bankrupt','Bankrupt'],digits=4,zero_division=0)
    (directory/'training_oof_report.txt').write_text(report,encoding='utf-8')
print('Training CV complete. These scores become selection scores if used to choose a scenario.')
display(pd.DataFrame(rows).round(2))


## 6. Refit and report on the test set
For each feature set, refit on all80% training companies and evaluate on the original20% test companies. Metrics retain their familiar names; Bankruptcy=1 is the positive class. A score>=0.5 predicts Bankruptcy. Exported predictions permit later threshold/curve audits; this notebook does not export or deploy a replacement production model.

In [ ]:
# Read validated OOF summary so this cell can be rerun without duplicating rows.
rows=pd.read_csv(OUT/'results.csv',float_precision='round_trip').query("Stage == 'Training OOF'").to_dict('records')
for name,columns in SCENARIOS.items():
    directory=OUT/name
    checkpoint=directory/'test_predictions.npz'
    if checkpoint.exists():
        scores=checked_predictions(checkpoint,test_idx,y_test)
    else:
        pipe=make_model(SMOTE_RATIO)
        pipe.fit(X_train[columns],y_train)
        scores=pipe.predict_proba(X_test[columns])[:,1]
        np.savez_compressed(checkpoint,row_index=test_idx,y_true=y_test,y_score=scores)
    reference=next(r for r in rows if r['Scenario']==name and r['Stage']=='Training OOF')
    row={**reference,'Stage':'Test',**score_metrics(y_test,scores)}
    rows.append(row)
    report=f'=== Classification Report: RandomForest | 80-20 + CV=10 | SMOTE={SMOTE_RATIO} | {name} ===\n'
    report+=classification_report(y_test,(scores>=.5).astype(int),labels=[0,1],target_names=['Not Bankrupt','Bankrupt'],digits=4,zero_division=0)
    report+='\n'+' | '.join(f'{k}: {row[k]:.2f}' for k in ['Accuracy (%)','Precision (%)','Recall (%)','F1-Score (%)','ROC-AUC (%)'])
    print(report)
    (directory/'test_report.txt').write_text(report,encoding='utf-8')
    fig,ax=plt.subplots(figsize=(5.4,4.5))
    ConfusionMatrixDisplay.from_predictions(y_test,(scores>=.5).astype(int),labels=[0,1],display_labels=['Not Bankrupt','Bankrupt'],ax=ax,cmap='Blues',colorbar=False)
    ax.set_title(f'{name} | SMOTE {SMOTE_RATIO}')
    fig.tight_layout()
    fig.savefig(directory/'test_confusion_matrix.svg',bbox_inches='tight')
    fig.savefig(directory/'test_confusion_matrix.png',dpi=180,bbox_inches='tight')
    plt.show()
    fig,axes=plt.subplots(1,2,figsize=(11,4.5))
    RocCurveDisplay.from_predictions(y_test,scores,ax=axes[0])
    PrecisionRecallDisplay.from_predictions(y_test,scores,ax=axes[1])
    fig.suptitle(f'{name} | test | SMOTE {SMOTE_RATIO}')
    fig.tight_layout()
    fig.savefig(directory/'test_roc_pr.png',dpi=180,bbox_inches='tight')
    fig.savefig(directory/'test_roc_pr.svg',bbox_inches='tight')
    plt.show()
    pd.DataFrame(rows).to_csv(OUT/'results.csv',index=False)
results=pd.DataFrame(rows)
(OUT/'RESULTS.txt').write_text('SHIELD: no imputation; retrained reduced-input Random Forests.\n'+
    '80:20 split, training CV=10, fixed parameters, seed42, threshold0.5.\n'+
    'Previously inspected holdout; exploratory results.\n\n'+results.to_string(index=False),encoding='utf-8')
display(results.round(2))


## 7. Compare with the20-feature baseline
Changes in percentage metrics are percentage points; FP/FN changes are company counts. No automatic winner is chosen from the historical test set. Inspect training OOF alongside holdout results.

In [ ]:
comparison=results[results.Stage=='Test'].copy()
baseline=comparison[comparison.Scenario=='baseline_20'].iloc[0]
for metric in ['Precision (%)','Recall (%)','F1-Score (%)','F2 (%)','FP','FN']:
    comparison['Change '+metric]=comparison[metric]-baseline[metric]
comparison.to_csv(OUT/'test_changes_vs_baseline.csv',index=False)
# Keep original scenario order. Do not automatically pick a winner from the test set.
cols=['Scenario','Features','Removed','Precision (%)','Recall (%)','F1-Score (%)','FP','FN',
    'Change Precision (%)','Change Recall (%)','Change F1-Score (%)']
display(comparison[cols].round(2))
fig,ax=plt.subplots(figsize=(12,8))
comparison.set_index('Scenario')[['Precision (%)','Recall (%)','F1-Score (%)']].plot.barh(ax=ax)
ax.set_xlabel('Score (%)'); ax.set_xlim(0,100)
ax.set_title(f'Seven versus twenty inputs: RF | test | SMOTE {SMOTE_RATIO}')
fig.tight_layout()
fig.savefig(OUT/'test_comparison.svg',bbox_inches='tight')
fig.savefig(OUT/'test_comparison.png',dpi=180,bbox_inches='tight')
plt.show()
print('All outputs:',OUT)


## Reading the outcome and running the second ratio

- Compare seven against20 **within the same ratio** first. Higher recall means fewer missed bankruptcies; higher precision means fewer false alarms among positive alerts.
- Then change SMOTE_RATIO to1.0 in Section2 and rerun from there. Settings hashes isolate outputs; do not edit only the final display cell.
- On the historical test set with44 bankruptcies, one company corresponds to2.27 percentage points of recall. Small changes do not establish a meaningful winner.
- Removing columns tests a **retrained seven-input model**, not an existing20-input model receiving blank values. Supporting arbitrary missing subsets needs separate evaluation.
- IG is intentionally omitted from this fixed-subset comparison. A later experiment could apply IG inside training folds to a larger accounting-eligible pool.
- Formula verification, correct raw-to-model transformation, actual SME document checks and independent population validation remain separate steps before deployment.

Output files include `results.csv`, `RESULTS.txt`, `test_changes_vs_baseline.csv`, historical IG references, split/fold membership, classification reports, confusion matrices, ROC/precision-recall charts and prediction checkpoints. No datasets or production model files are changed.
